# Preprocessing of the raw final map

## 1. Loading libraries, parameters and data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json

from joblib import Parallel, delayed, effective_n_jobs

In [ ]:
from ddae1d.utils import despike_iterative, baseline_correction, percentile_offset
from ddae1d.paths import PROJECT_ROOT

### Loading parameters

Modify them in `./config.json` if needed.

In [ ]:
with open("config.json", "r") as f:
    config = json.load(f)

In [ ]:
raw_finalmap_filename = config["raw_finalmap_filename"]

trim_spectra = config["trim_spectra"]
spectral_ends = config["spectral_ends"]

apply_offset = config["apply_offset"]
offset_percentile = config["offset_percentile"]

apply_despike = config["apply_despike"]
n_jobs_despike = config["n_jobs_despike"]
despike_params = config["despike_params"]

apply_baseline_correction = config["apply_baseline_correction"]
polyfit_degree = config["polyfit_degree"]

### Loading raw training data

In [ ]:
raw_finalmap = np.load(PROJECT_ROOT / "data" / "raw" / "final-map" / raw_finalmap_filename)

In [ ]:
n_x, n_y, _ = raw_finalmap.shape

In [ ]:
raw_finalmap = raw_finalmap.reshape((n_x * n_y, -1))

## 2. Trimming spectral ends and substracting offset

### *If needed, trim the spectral ends to remove artefacts such as Rayleigh scattering*

Spectra must be trimmed in the same way as the training data.

Modify the values of `trim_spectra` and `spectral_ends` in `./config.json` if needed.

In [ ]:
if trim_spectra:
    # Trim spectral ends
    finalmap = raw_finalmap[:, spectral_ends[0]:spectral_ends[1]]
else:
    finalmap = raw_finalmap

In [ ]:
del raw_finalmap

### *If relevant, subtract an 'offset' baseline - common to the whole dataset. This should be done from the raw spectra before further preprocessing.*

#### Method for computing the offset: low per-channel percentile across spatial points
For each spectral channel, the offset is the `offset_percentile`-th percentile of the intensities across spatial points. It is robust to a few anomalous spectra and does not inject the noise of a single spectrum into every spectrum.

*Tune `offset_percentile` in `config.json`*

In [ ]:
# If enabled, subtract the offset profile (low per-channel percentile across spatial points) from all spectra
if apply_offset:
    offset_profile = percentile_offset(finalmap, offset_percentile)

    # Plot the offset profile for visual inspection
    plt.figure(figsize=(10, 6))
    plt.plot(offset_profile)
    plt.xlabel("Spectral Channel")
    plt.ylabel("Intensity (a.u.)")
    plt.title(f"Offset profile (percentile {offset_percentile} across spatial points)")
    plt.xlim(0, len(offset_profile) - 1)
    plt.tight_layout()
    plt.show()

    finalmap_after_offset = finalmap - offset_profile
else:
    finalmap_after_offset = finalmap

In [ ]:
del finalmap

## 3. Despiking (parallel processing)

Tune the number of parallel jobs: `n_jobs_despike` in `config.json`

In [ ]:
def despike_task(point):
    despiked_spectrum, spikes_spectrum = despike_iterative(
        finalmap_after_offset[point, :],
        **despike_params
    )
    return (point, despiked_spectrum, spikes_spectrum)

In [ ]:
if apply_despike:
    print(f"Number of jobs for parallel despiking: {effective_n_jobs(n_jobs_despike)}")
    spikes = np.zeros_like(finalmap_after_offset)
    finalmap_after_despike = np.zeros_like(finalmap_after_offset)
    print("Starting parallel despiking of final map...")
    print(f"Total spectra to despike (tasks to do): {finalmap_after_offset.shape[0]}")
    results = Parallel(n_jobs=n_jobs_despike, verbose=1)(
        delayed(despike_task)(point)
        for point in range(finalmap_after_offset.shape[0])
    )
    for point, despiked_spectrum, spikes_spectrum in results:
        finalmap_after_despike[point, :] = despiked_spectrum
        spikes[point, :] = spikes_spectrum
    del results
    nonzero_indices = np.flatnonzero(np.any(spikes != 0, axis=-1))
    percentage_nonzero = len(nonzero_indices) / (finalmap_after_despike.shape[0]) * 100
    print(f"Percentage of spectra despiked: {percentage_nonzero:.2f}%")
    print("Number of spectra despiked:", len(nonzero_indices))
else:
    finalmap_after_despike = finalmap_after_offset

In [ ]:
del finalmap_after_offset

### Plot 20 random examples of despiked spectra vs removed spikes
Verify that despiking is effective and does not remove relevant signal

In [ ]:
if apply_despike:
    n_plot = min(20, len(nonzero_indices))
    selected_indices = nonzero_indices[np.random.choice(len(nonzero_indices), n_plot, replace=False)]

    fig, axes = plt.subplots(4, 5, figsize=(20, 12), sharex=True, sharey=False)
    axes = axes.flatten()

    for ax, point in zip(axes, selected_indices):
        ax.plot(finalmap_after_despike[point], label='Despiked', zorder=1)
        ax.plot((finalmap_after_despike + spikes)[point], label='Removed Spikes', zorder=0)
        ax.set_title(f'Point {point}')
        ax.legend()

    plt.tight_layout()
    plt.show()

## 4. Baseline correction

In [ ]:
if apply_baseline_correction:
    finalmap_after_baseline = baseline_correction(finalmap_after_despike, polyfit_degree)
else:
    finalmap_after_baseline = finalmap_after_despike

In [ ]:
del finalmap_after_despike

## 5. Saving preprocessed training data

In [ ]:
finalmap_reshaped = finalmap_after_baseline.reshape((n_x, n_y, -1))

In [ ]:
np.save(PROJECT_ROOT / "data" / "preprocessed" / "final-map" / "preprocessed-final-map.npy", finalmap_reshaped)